# 02 — Entrenamiento y Selección del Mejor Modelo

**Proyecto:** Detección de jugadores para offside en fútbol  
**Dataset:** Roboflow `team-separation` v5 — 1.200 imágenes 640×640, 7 clases, ~21.600 bboxes  
**Notebook anterior:** `01_dataset_preparation.ipynb`  
**Hardware asumido:** Google Colab T4 GPU (16 GB VRAM)  

## Objetivo

Fine-tunear un detector de objetos YOLOv8 sobre el dataset de broadcast de fútbol para localizar jugadores (TEAM 1, TEAM 2), árbitros, arqueros, pelota, arco y corners — información necesaria para trazar la línea de offside.

## Reproducibilidad

Se fija `seed=42` en todos los entrenamientos. Con `deterministic=True` se desactivan operaciones no deterministas de CUDA. **Nota:** Ultralytics activa mosaic augmentation que tiene componentes estocásticos aun con semilla fija — esto es esperado y está documentado.

## Experimentos

| Exp | Modelo | Estrategia | Dimensión |
|-----|--------|-----------|----------|
| 1 | YOLOv8s | Full fine-tune | Baseline |
| 2 | YOLOv8s | Freeze backbone (`freeze=10`) | Estrategia de fine-tuning |
| 3 | YOLOv8m | Full fine-tune + class weights | Capacidad + class imbalance |

---
## 1. Configuración del entorno

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3.0", "roboflow>=1.1.0"])

In [ ]:
import os, sys, shutil, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import yaml
import cv2
from PIL import Image
from IPython.display import display

import torch
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
import ultralytics; print(f"Ultralytics: {ultralytics.__version__}")

In [ ]:
if torch.cuda.is_available():
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("ADVERTENCIA: GPU no disponible. Usar Colab: Runtime > Change runtime type > GPU")

In [ ]:
SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Semilla fijada: {SEED}")

---
## 2. Acceso al dataset y configuración de rutas

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_ROOT = Path("/content/ProyectoFinalRedesOffside")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/ElPacho21/ProyectoFinalRedesOffside.git",
                        str(REPO_ROOT)], check=True)
    os.chdir(REPO_ROOT)
    sys.path.insert(0, str(REPO_ROOT))
else:
    if Path.cwd().name == "dev":
        os.chdir(Path.cwd().parent)

print(f"Directorio de trabajo: {Path.cwd()}")

In [ ]:
RAW_DIR = Path("data/raw")
IMAGES_TRAIN = RAW_DIR / "train" / "images"

if not IMAGES_TRAIN.exists() or not any(IMAGES_TRAIN.glob("*.jpg")):
    print("Descargando dataset desde Roboflow...")
    # Pasar API key como variable de entorno al subprocess
    if IN_COLAB:
        from google.colab import userdata
        api_key = userdata.get("ROBOFLOW_API_KEY")
    else:
        api_key = os.environ.get("ROBOFLOW_API_KEY", "")
    env = {**os.environ, "ROBOFLOW_API_KEY": api_key}
    subprocess.run([sys.executable, "data/download_dataset.py"], check=True, env=env)
else:
    print(f"Dataset presente: {sum(1 for _ in IMAGES_TRAIN.glob('*.jpg'))} imágenes en train/images")

In [ ]:
YAML_PATH = RAW_DIR / "data.yaml"
assert YAML_PATH.exists(), f"No encontrado: {YAML_PATH}"

with open(YAML_PATH) as f:
    yaml_data = yaml.safe_load(f)

yaml_data["path"] = str(RAW_DIR.resolve())
yaml_data["train"] = "train/images"
yaml_data["val"]   = "valid/images"
yaml_data["test"]  = "test/images"

with open(YAML_PATH, "w") as f:
    yaml.dump(yaml_data, f, allow_unicode=True)

DATA_YAML = str(YAML_PATH.resolve())
CLASS_NAMES = yaml_data["names"]
print(f"data.yaml configurado → {DATA_YAML}")
print(f"Clases ({len(CLASS_NAMES)}): {CLASS_NAMES}")

In [ ]:
train_df = pd.read_csv("data/train.csv")
val_df   = pd.read_csv("data/val.csv")
test_df  = pd.read_csv("data/test.csv")

counts_train = train_df["class_name"].value_counts().rename("train")
counts_val   = val_df["class_name"].value_counts().rename("val")
counts_test  = test_df["class_name"].value_counts().rename("test")
imbalance_df = pd.concat([counts_train, counts_val, counts_test], axis=1).fillna(0).astype(int)
imbalance_df["total"] = imbalance_df.sum(axis=1)
imbalance_df = imbalance_df.sort_values("total", ascending=False)

print("=== Distribución de anotaciones por clase ===")
display(imbalance_df)

fig, ax = plt.subplots(figsize=(10, 4))
imbalance_df["train"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Instancias por clase (train)")
ax.set_xlabel("Clase")
ax.set_ylabel("Cantidad de bboxes")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

---
## 3. Justificación del modelo preentrenado

### ¿Por qué YOLOv8?

YOLOv8 (Ultralytics, 2023) es un detector single-stage anchor-free que ofrece:
- **Pesos preentrenados en COCO** — COCO incluye `person` (≈ jugadores/árbitros) y `sports ball` (≈ pelota). Dominio de origen altamente similar → features del backbone directamente útiles.
- **Desacople cabeza de detección** — permite fine-tunear la head sin tocar el backbone, clave para la estrategia de freeze.
- **Integración Ultralytics** — métricas automáticas (mAP@50, mAP@50-95, P, R, confusion matrix), API de extensión via `DetectionTrainer`.

### Escala de modelos en los experimentos

YOLOv8 ofrece variantes n < s < m < l < x. Se eligieron `s` y `m` dentro de la misma familia arquitectónica:
- **YOLOv8s** (11M params): baseline razonable para dataset chico, menor riesgo de overfitting.
- **YOLOv8m** (25M params): mayor capacidad sin exceder lo que el dataset puede sostener sin overfit severo.

### Class imbalance — Class weights

El dataset tiene fuerte desbalance (TEAM 1/TEAM 2 dominan; Goal_Net y Corner son clases raras). Para el Exp 3 se implementa un `DetectionTrainer` custom (`src/weighted_trainer.py`) que inyecta `pos_weight` per-clase en la BCE de clasificación:

$$w_i = \frac{N_{total}}{n_{clases} \cdot N_i}$$

Clases raras reciben mayor peso → gradiente proporcionalmente mayor. Es el mecanismo estándar de weighted BCE, aplicado a la componente de clasificación de YOLOv8.

### Augmentations: Ultralytics vs Albumentations

En el notebook `01_dataset_preparation.ipynb` se definió un pipeline Albumentations (`src/dataset.py:73`). **En Phase 3 ese pipeline NO se usa** — Ultralytics maneja sus propias augmentations internamente. `hsv_h=0.015` (default) es deliberadamente estrecho para preservar colores de camisetas TEAM 1/TEAM 2.

---
## 4. Experimento 1 — YOLOv8s Full Fine-Tune (Baseline)

**Configuración:**
- Modelo: `yolov8s.pt` (COCO pretrained, 11M params)
- Estrategia: full fine-tune (todos los parámetros actualizables)
- Optimizador: AdamW, lr=1e-3, cosine LR scheduler
- 50 épocas, early stopping patience=10, batch=16

**AdamW + cosine:** AdamW desacopla L2 regularization del gradient update. Cosine scheduler reduce LR suavemente → mejor convergencia en fine-tuning que step decay.

In [ ]:
set_seed(SEED)
model1 = YOLO("yolov8s.pt")

t0 = time.time()
results1 = model1.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=1e-3,
    cos_lr=True,
    patience=10,
    seed=SEED,
    deterministic=True,
    project="runs/exp",
    name="exp1_yolov8s_full",
    exist_ok=True,
    verbose=False,
)
t1 = time.time()
print(f"\nExp 1 completado en {(t1 - t0) / 60:.1f} minutos")

del model1
torch.cuda.empty_cache()

---
## 5. Experimento 2 — YOLOv8s Freeze Backbone

**Configuración:** Idéntica al Exp 1, salvo `freeze=10`.

**¿Qué se congela?** `freeze=10` bloquea las primeras 10 capas (backbone CSPDarknet). Solo neck (PANet) y head se entrenan.

**Hipótesis:** con 1.200 imágenes, el backbone COCO ya tiene features útiles (personas = jugadores). Congelarlo reduce el espacio de parámetros a optimizar → menos overfitting. Se contrasta contra Exp 1 para determinar si el fine-tuning completo aporta o sobreescribe features útiles.

In [ ]:
set_seed(SEED)
model2 = YOLO("yolov8s.pt")

t0 = time.time()
results2 = model2.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=1e-3,
    cos_lr=True,
    patience=10,
    freeze=10,
    seed=SEED,
    deterministic=True,
    project="runs/exp",
    name="exp2_yolov8s_frozen",
    exist_ok=True,
    verbose=False,
)
t1 = time.time()
print(f"\nExp 2 completado en {(t1 - t0) / 60:.1f} minutos")

del model2
torch.cuda.empty_cache()

---
## 6. Experimento 3 — YOLOv8m Full Fine-Tune + Class Weights

**Configuración:**
- Modelo: `yolov8m.pt` (COCO pretrained, 25M params — mayor capacidad)
- Class weights per-clase calculados desde frecuencias del train set
- `batch=8` por mayor consumo de VRAM con YOLOv8m

**Class weights:** se usa `src/weighted_trainer.py`, que subclasea `DetectionTrainer` e inyecta `pos_weight=tensor([w0..w6])` en la `BCEWithLogitsLoss` de clasificación. El peso de cada clase es inversamente proporcional a su frecuencia:

$$w_i = \frac{N_{total}}{n_{clases} \cdot N_i}$$

Esto incrementa el gradiente de clases raras (Goal_Net, Corner) sin modificar la regresión de bboxes.

**Dimensiones comparadas vs Exp 1:** capacidad del modelo (s vs m) + manejo de imbalance (sin/con weights). Para atribuir mejoras: si clases raras suben desproporcionalmente → weights; si todas mejoran uniforme → capacidad.

In [ ]:
from src.weighted_trainer import build_weighted_trainer

# Calcular class counts desde train.csv (mismo orden que CLASS_NAMES / data.yaml)
class_counts = [
    int(train_df[train_df["class_name"] == cls].shape[0])
    for cls in CLASS_NAMES
]
print(f"Conteos por clase (orden data.yaml): {dict(zip(CLASS_NAMES, class_counts))}")

device = "cuda" if torch.cuda.is_available() else "cpu"
WeightedTrainer = build_weighted_trainer(class_counts, device=device)

In [ ]:
set_seed(SEED)
model3 = YOLO("yolov8m.pt")

t0 = time.time()
results3 = model3.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=1e-3,
    cos_lr=True,
    patience=10,
    seed=SEED,
    deterministic=True,
    project="runs/exp",
    name="exp3_yolov8m_weighted",
    exist_ok=True,
    verbose=False,
    trainer=WeightedTrainer,
)
t1 = time.time()
print(f"\nExp 3 completado en {(t1 - t0) / 60:.1f} minutos")

del model3
torch.cuda.empty_cache()

---
## 7. Tabla comparativa de experimentos

In [ ]:
def find_run_dir(run_name):
    matches = list(Path("runs").rglob(f"{run_name}/results.csv"))
    if matches:
        return matches[0].parent
    raise FileNotFoundError(f"No encontrado results.csv para {run_name}")

def load_results(run_name):
    run_dir = find_run_dir(run_name)
    df = pd.read_csv(run_dir / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    map50_col     = next(c for c in df.columns if "mAP50(B)" in c or "mAP_0.5" in c)
    map5095_col   = next(c for c in df.columns if "mAP50-95(B)" in c or "mAP_0.5:0.95" in c)
    precision_col = next(c for c in df.columns if "precision" in c.lower())
    recall_col    = next(c for c in df.columns if "recall" in c.lower())
    best_idx = df[map5095_col].idxmax()
    best = df.loc[best_idx]
    time_cols = [c for c in df.columns if "time" in c.lower()]
    train_time = round(df[time_cols[0]].sum() / 60, 1) if time_cols else None
    return {
        "run":        run_name,
        "best_epoch": int(best_idx) + 1,
        "mAP@50":     round(float(best[map50_col]),     4),
        "mAP@50-95":  round(float(best[map5095_col]),   4),
        "Precision":  round(float(best[precision_col]), 4),
        "Recall":     round(float(best[recall_col]),    4),
        "tiempo_min": train_time,
    }

exp_names = ["exp1_yolov8s_full", "exp2_yolov8s_frozen", "exp3_yolov8m_weighted"]
rows = [load_results(n) for n in exp_names]
comparison_df = pd.DataFrame(rows).set_index("run")
comparison_df.index = ["Exp1: YOLOv8s full", "Exp2: YOLOv8s freeze", "Exp3: YOLOv8m weighted"]
print("=== Tabla comparativa de experimentos ===")
display(comparison_df)
best_run_name = exp_names[comparison_df["mAP@50-95"].values.argmax()]
print(f"\nMejor experimento (mAP@50-95): {best_run_name}")

### Análisis comparativo

> **Completar luego de ver los resultados.** Guía:
> - **Exp1 vs Exp2:** Si freeze supera full → backbone COCO suficiente para 1.200 imgs. Si full gana → features broadcast (camisetas, perspectiva) justifican actualizar todo el backbone.
> - **Exp3 vs Exp1:** Si YOLOv8m gana en clases raras (Goal_Net, Corner) → class weights aportaron. Si gana uniforme → fue la capacidad extra del modelo m.

---
## 8. Curvas de entrenamiento

In [ ]:
def plot_training_curves(run_name, ax_row, title_prefix):
    run_dir = find_run_dir(run_name)
    df = pd.read_csv(run_dir / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    epochs = range(1, len(df) + 1)
    train_loss_cols = [c for c in df.columns if "train" in c.lower() and "loss" in c.lower()]
    val_loss_cols   = [c for c in df.columns if "val" in c.lower() and "loss" in c.lower()]
    map50_col   = next(c for c in df.columns if "mAP50(B)" in c or "mAP_0.5" in c)
    map5095_col = next(c for c in df.columns if "mAP50-95(B)" in c or "mAP_0.5:0.95" in c)
    ax_loss = ax_row[0]
    if train_loss_cols:
        ax_loss.plot(epochs, df[train_loss_cols].sum(axis=1), label="Train loss", color="steelblue")
    if val_loss_cols:
        ax_loss.plot(epochs, df[val_loss_cols].sum(axis=1), label="Val loss", color="tomato", linestyle="--")
    ax_loss.set_title(f"{title_prefix}\nLoss (train + val)")
    ax_loss.set_xlabel("Época")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend()
    ax_loss.grid(True, alpha=0.3)
    ax_map = ax_row[1]
    ax_map.plot(epochs, df[map50_col],   label="mAP@50",    color="green")
    ax_map.plot(epochs, df[map5095_col], label="mAP@50-95", color="darkgreen", linestyle="--")
    ax_map.set_title(f"{title_prefix}\nmAP (validación)")
    ax_map.set_xlabel("Época")
    ax_map.set_ylabel("mAP")
    ax_map.legend()
    ax_map.grid(True, alpha=0.3)

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
labels = [
    ("exp1_yolov8s_full",     "Exp1: YOLOv8s Full"),
    ("exp2_yolov8s_frozen",   "Exp2: YOLOv8s Freeze"),
    ("exp3_yolov8m_weighted", "Exp3: YOLOv8m Weighted"),
]
for i, (run, label) in enumerate(labels):
    plot_training_curves(run, axes[i], label)

plt.suptitle("Curvas de entrenamiento — Comparativa de experimentos", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("dev/training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Guardado: dev/training_curves.png")

### Análisis de overfitting / underfitting

> **Completar luego de ver las curvas.** Guía:
> - **Overfitting:** train loss baja, val loss sube/estanca. Early stopping activo.
> - **Underfitting:** ambas losses altas al final. mAP val < 0.4.
> - **Buen fit:** train y val loss descienden juntas hasta convergencia.

---
## 9. Selección del mejor modelo

In [ ]:
best_run_name = exp_names[comparison_df["mAP@50-95"].values.argmax()]
BEST_WEIGHTS = find_run_dir(best_run_name) / "weights" / "best.pt"

assert BEST_WEIGHTS.exists(), f"No encontrado: {BEST_WEIGHTS}"
print(f"Mejor modelo: {best_run_name}")
print(f"Pesos:        {BEST_WEIGHTS}")
print(f"mAP@50-95 (val): {comparison_df['mAP@50-95'].max():.4f}")

---
## 10. Evaluación final en el conjunto de test

**Importante:** Se evalúa sobre **test** (nunca visto). Las métricas de validación se usaron solo para selección de modelo.

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
test_metrics = best_model.val(
    data=DATA_YAML,
    split="test",
    plots=True,
    save_json=True,
    verbose=False,
)
print(f"\nmAP@50     (test): {test_metrics.box.map50:.4f}")
print(f"mAP@50-95  (test): {test_metrics.box.map:.4f}")

In [ ]:
names = test_metrics.names
n_classes = len(names)
p  = test_metrics.box.p
r  = test_metrics.box.r
f1 = 2 * p * r / (p + r + 1e-9)
ap50   = test_metrics.box.ap50
ap5095 = test_metrics.box.maps

per_class_df = pd.DataFrame({
    "Clase":     [names[i] for i in range(n_classes)],
    "Precision": p.round(4),
    "Recall":    r.round(4),
    "F1":        f1.round(4),
    "mAP@50":    ap50.round(4),
    "mAP@50-95": ap5095.round(4),
}).set_index("Clase")

print("=== Métricas por clase (test set) ===")
display(per_class_df)

fig, ax = plt.subplots(figsize=(10, 4))
per_class_df["F1"].sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.axvline(per_class_df["F1"].mean(), color="red", linestyle="--", label="Media F1")
ax.set_title("F1 por clase (test set)")
ax.set_xlabel("F1")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar matriz de confusión generada por Ultralytics
cm_paths = list(Path("runs").rglob("confusion_matrix*.png"))
if cm_paths:
    for cm_path in sorted(cm_paths)[:2]:
        img = Image.open(cm_path)
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.title(cm_path.name)
        plt.tight_layout()
        plt.show()
else:
    print("Matriz de confusión no encontrada. Verificar plots=True en model.val().")

---
## 11. Análisis de detecciones

In [ ]:
TEST_IMAGES_DIR = Path("data/raw/test/images")
TEST_LABELS_DIR = Path("data/raw/test/labels")
IOU_THRESHOLD  = 0.5
CONF_THRESHOLD = 0.25
MAX_BOXES_SHOWN = 15  # límite de boxes por imagen para no saturar visualización

def load_gt_boxes(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        xc, yc, w, h = map(float, parts[1:5])
        x1 = (xc - w / 2) * img_w
        y1 = (yc - h / 2) * img_h
        x2 = (xc + w / 2) * img_w
        y2 = (yc + h / 2) * img_h
        boxes.append((cls, x1, y1, x2, y2))
    return boxes

def iou(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a[1], box_a[2], box_a[3], box_a[4]
    xb1, yb1, xb2, yb2 = box_b[1], box_b[2], box_b[3], box_b[4]
    inter_x1 = max(xa1, xb1); inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2); inter_y2 = min(ya2, yb2)
    inter = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    return inter / (area_a + area_b - inter + 1e-6)

error_cases = []
good_cases  = []

test_imgs = sorted(TEST_IMAGES_DIR.glob("*.jpg"))[:500]

for img_path in test_imgs:
    img = Image.open(img_path)
    w, h = img.size
    gt = load_gt_boxes(TEST_LABELS_DIR / (img_path.stem + ".txt"), w, h)
    if not gt:
        continue
    result = best_model.predict(source=str(img_path), conf=CONF_THRESHOLD, verbose=False, save=False)[0]
    preds = []
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            cls_id = int(box.cls.item())
            conf   = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            preds.append((cls_id, x1, y1, x2, y2, conf))
        # Ordenar por confianza descendente y limitar
        preds.sort(key=lambda x: x[5], reverse=True)
        preds = preds[:MAX_BOXES_SHOWN]

    has_error = False
    all_matched = True
    for gt_box in gt:
        matched = False
        wrong_class = False
        for pred in preds:
            pred_box_fmt = (pred[0], pred[1], pred[2], pred[3], pred[4])
            if iou(gt_box, pred_box_fmt) > IOU_THRESHOLD:
                matched = True
                if pred[0] != gt_box[0]:
                    wrong_class = True
                break
        if wrong_class:
            has_error = True
            error_cases.append((img_path, gt, preds, "clase_errónea"))
            break
        if not matched:
            has_error = True
            all_matched = False
            error_cases.append((img_path, gt, preds, "falso_negativo"))
            break
    if not has_error and len(preds) > 0:
        good_cases.append((img_path, gt, preds))

print(f"Errores: {len(error_cases)} / {len(test_imgs)} imágenes analizadas")
print(f"Correctas: {len(good_cases)}")

In [ ]:
def draw_boxes(ax, img_path, gt, preds, title):
    img_arr = np.array(Image.open(img_path))
    ax.imshow(img_arr)
    for (cls_id, x1, y1, x2, y2) in gt:
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 3, 0), CLASS_NAMES[cls_id],
                color="lime", fontsize=6, backgroundcolor="black")
    for (cls_id, x1, y1, x2, y2, conf) in preds:
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=1.5, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, min(y2 + 10, img_arr.shape[0] - 1),
                f"{CLASS_NAMES[cls_id]} {conf:.2f}",
                color="red", fontsize=6, backgroundcolor="white")
    ax.set_title(title, fontsize=8)
    ax.axis("off")

# --- Errores ---
n_err = min(9, len(error_cases))
if n_err > 0:
    cols = 3
    rows = (n_err + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 6))
    axes = axes.flatten()
    for i, (img_path, gt, preds, error_type) in enumerate(error_cases[:n_err]):
        draw_boxes(axes[i], img_path, gt, preds,
                   f"{img_path.name}\nError: {error_type}")
    for j in range(n_err, len(axes)):
        axes[j].axis("off")
    plt.suptitle("Detecciones con error — GT (verde) vs Pred (rojo)", fontsize=12)
    plt.tight_layout()
    plt.savefig("dev/error_analysis.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Guardado: dev/error_analysis.png")
else:
    print("No se encontraron errores con los criterios actuales.")

In [ ]:
# --- Detecciones correctas ---
n_good = min(9, len(good_cases))
if n_good > 0:
    cols = 3
    rows = (n_good + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 6))
    axes = axes.flatten()
    for i, (img_path, gt, preds) in enumerate(good_cases[:n_good]):
        draw_boxes(axes[i], img_path, gt, preds,
                   f"{img_path.name}\n✓ Detección correcta")
    for j in range(n_good, len(axes)):
        axes[j].axis("off")
    plt.suptitle("Detecciones correctas — GT (verde) vs Pred (rojo)", fontsize=12)
    plt.tight_layout()
    plt.savefig("dev/good_detections.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Guardado: dev/good_detections.png")
else:
    print("No se encontraron casos completamente correctos con los criterios actuales.")

### Causas posibles de error

> **Completar luego de ver las imágenes.** Causas típicas esperadas:
> - **TEAM 1 / TEAM 2 confundidos:** camisetas de colores similares o iluminación variable.
> - **Ball no detectada:** objeto más chico del dataset, sensible a motion blur de broadcast.
> - **Goal_Net / Corner falsos negativos:** clases con pocas instancias en train.
> - **Oclusión:** jugadores superpuestos → una sola bbox predicha donde GT tiene dos separadas.

---
## 12. Guardar modelo final

In [ ]:
DEST_MODEL = Path("dev/modelo.pth")
shutil.copy(BEST_WEIGHTS, DEST_MODEL)

size_mb = DEST_MODEL.stat().st_size / 1e6
print(f"Modelo guardado: {DEST_MODEL}")
print(f"Tamaño: {size_mb:.1f} MB")

test_load = YOLO(str(DEST_MODEL))
print(f"Carga OK: {type(test_load.model).__name__}")
print("NOTA: cargar con YOLO('dev/modelo.pth'), no con torch.load() directo.")
del test_load

---
## 13. Verificación de requisitos

| Requisito | Sección | Estado |
|-----------|---------|--------|
| a) Modelo preentrenado + justificación + modificaciones + estrategia freeze | Sec. 3, Sec. 4-6 | ✅ |
| b) Loss, optimizador, LR, scheduler, épocas, batch, hardware, tiempo | Sec. 4-6 | ✅ |
| c) ≥3 experimentos + tabla comparativa + análisis | Sec. 4-7 | ✅ |
| d) Curvas loss+mAP train/val + comentario overfitting/underfitting | Sec. 8 | ✅ |
| e) Métricas test: P/R/F1 por clase + matriz de confusión | Sec. 10 | ✅ |
| f) Análisis de errores con ejemplos visuales | Sec. 11 | ✅ |
| g) Modelo guardado como `dev/modelo.pth` | Sec. 12 | ✅ |
| h) Notebook en `dev/02_model_training.ipynb` | — | ✅ |
| i) Reproducible end-to-end (seed, deterministic) | Sec. 1, Sec. 4-6 | ✅ |